##  Installation Set up

In [ ]:
!pip install llama-index pymupdf llama-index-embeddings-huggingface

## Install Libraries

In [ ]:
from llama_index.core import VectorStoreIndex, Document
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.settings import Settings
import fitz  # PyMuPDF
import time

## Upload Document



In [ ]:
# Uploaded file
pdf_path = "/content/sample_contract_1.pdf"
doc = fitz.open(pdf_path)
text = "\\n".join([page.get_text() for page in doc])

print(f"Extracted {len(text.split())} words from the contract.")

Extracted 315 words from the contract.


In [ ]:
# Create & Configure the embedding model
embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
Settings.embed_model = embed_model

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# Define a sentence splitter (can also use TokenTextSplitter or CharacterTextSplitter)
text_splitter = SentenceSplitter(chunk_size=50, chunk_overlap=50)

# Turn raw text into a list of Document objects
documents = [Document(text=text)]

# Convert into nodes (smaller chunks)
nodes = text_splitter.get_nodes_from_documents(documents)

# Then create the index from these nodes
index = VectorStoreIndex(nodes)

## Experiment with Top-K Retrieval

In [ ]:
query = "What is the maximum loan amount a borrower can apply for?"
top_k_values = [2, 5, 10]

for top_k in top_k_values:
    print(f"\\n--- Results for top_k = {top_k} ---")
    retriever = index.as_retriever(similarity_top_k=top_k)
    nodes = retriever.retrieve(query)
    for i, node in enumerate(nodes):
        print(f"Result {i+1}:")
        print(node.get_text())
        print("-" * 80)

\n--- Results for top_k = 2 ---
Result 1:
Payment terms are
net 30 days from receipt of invoice.
2.3 Late payments shall bear interest at the rate of 1.5% per month from the due date until paid in full.
3.
--------------------------------------------------------------------------------
Result 2:
3. TERM AND TERMINATION
3.1 This Agreement shall commence on the Effective Date and shall continue for a period of one (1)
year, unless earlier terminated as provided herein.
--------------------------------------------------------------------------------
\n--- Results for top_k = 5 ---
Result 1:
Payment terms are
net 30 days from receipt of invoice.
2.3 Late payments shall bear interest at the rate of 1.5% per month from the due date until paid in full.
3.
--------------------------------------------------------------------------------
Result 2:
3. TERM AND TERMINATION
3.1 This Agreement shall commence on the Effective Date and shall continue for a period of one (1)
year, unless earlier termin

# Apply Similarity Threshold

In [ ]:
# Retrieve nodes with top_k = 10
reteriver = index.as_retriever(similarity_top_k=10)
retrived_nodes = reteriver.retrieve(query)

In [50]:
# Apply thresholding on retrived node of top_k = 10
for threshold in [0.7, 0.75, 0.8]:
  filtered_nodes = [node for node in retrived_nodes if node.score and node.score > threshold ]
  print(f"Result for threshold = {threshold}")
  print(f"Filtered {len(filtered_nodes)} nodes out of {len(retrived_nodes)}")
  for i, node in enumerate(filtered_nodes):
    print(f"Result {i+1}, Score: {node.score: .2f}")
    print(node.get_text())
    print("-" * 80)


Result for threshold = 0.7
Filtered 0 nodes out of 10
Result for threshold = 0.75
Filtered 0 nodes out of 10
Result for threshold = 0.8
Filtered 0 nodes out of 10


# Combined Configurations

In [51]:
experiments = [
    {"top_k": 5, "threshold": None},
    {"top_k": 8, "threshold": 0.75},
    {"top_k": 5, "threshold": 0.8},
]

In [57]:
!pip install llama-index-postprocessor-sbert-rerank

In [58]:
from llama_index.postprocessor.sbert_rerank import SentenceTransformerRerank

reranker = SentenceTransformerRerank(
    model="cross-encoder/ms-marco-MiniLM-L-6-v2",
    top_n=3   # how many chunks to keep after reranking
)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [59]:
for experiment in experiments:
  print(f"Experiment top_k = {experiment['top_k']}, and Threshold = {experiment['threshold']}")
  reteriver = index.as_retriever(similarity_top_k = experiment['top_k'])
  nodes = reteriver.retrieve(query)


  # Reranking
  if len(nodes) > 0:
      nodes = reranker.postprocess_nodes(
          nodes,
          query_str=query
      )
  print(f"After reranking: {len(nodes)} chunks")

  # Threshold filtering
  if experiment["threshold"]:
    nodes = [node for node in nodes if node.score and node.score > experiment["threshold"]]
  print(f"Chunked Retrived: {len(nodes)}")

  # Results
  for i, node in enumerate(nodes):
    print(f"Result: {i+1} Chunk size: {node.score}")
    print(node.get_text())
    print("-" * 80)



Experiment top_k = 5, and Threshold = None
After reranking: 3 chunks
Chunked Retrived: 3
Result: 1 Chunk size: -10.25059700012207
Payment terms are
net 30 days from receipt of invoice.
2.3 Late payments shall bear interest at the rate of 1.5% per month from the due date until paid in full.
3.
--------------------------------------------------------------------------------
Result: 2 Chunk size: -10.882826805114746
3. TERM AND TERMINATION
3.1 This Agreement shall commence on the Effective Date and shall continue for a period of one (1)
year, unless earlier terminated as provided herein.
--------------------------------------------------------------------------------
Result: 3 Chunk size: -10.97486686706543
3.2 Either party may terminate this Agreement upon thirty (30) days written notice to the other party.
4.
--------------------------------------------------------------------------------
Experiment top_k = 8, and Threshold = 0.75
After reranking: 3 chunks
Chunked Retrived: 0
Experiment